In [8]:
# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import json
from collections import Counter

# Project root
PROJECT_DIR = Path.cwd()

# Dataset paths
DATASET_DIR = PROJECT_DIR / "dataset" / "seeing_through_fog"
DOWNLOAD_DIR = DATASET_DIR / "downloads"
EXTRACTED_DIR = DATASET_DIR / "extracted"

# Download metadata
JSON_PATH = PROJECT_DIR / "download_links.json"

# Results
OUTPUT_DIR = PROJECT_DIR / "results" / "stage1_camera"

print("PROJECT CONFIGURATION")
print("-" * 55)

print(f"Project directory: {PROJECT_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Download directory: {DOWNLOAD_DIR}")
print(f"Extracted directory: {EXTRACTED_DIR}")
print(f"JSON file: {JSON_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

# ============================================================
# 2. CREATE DATASET DIRECTORIES
# ============================================================

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DIRECTORY STATUS")
print("-" * 55)

print("Downloads folder:", DOWNLOAD_DIR.exists())
print("Extracted folder:", EXTRACTED_DIR.exists())
print("Results folder:", OUTPUT_DIR.exists())

# ============================================================
# 3. LOAD DOWNLOAD METADATA
# ============================================================

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Download metadata not found: {JSON_PATH}"
    )

with open(JSON_PATH, "r", encoding="utf-8") as file:
    dataset_info = json.load(file)

urls = dataset_info["urls"]

print("DATASET METADATA")
print("-" * 55)

print(f"Total download links: {len(urls)}")
print("JSON loaded successfully!")

# ============================================================
# 1.4 INSPECT AVAILABLE DATASET RESOURCES
# ============================================================

from collections import Counter

resource_counts = Counter()

for item in urls:

    # Extract the dataset path
    key = item["key"]

    # Split the path into components
    path_parts = key.strip("/").split("/")

    # Ignore the dataset root directory
    if len(path_parts) >= 2:

        resource_name = path_parts[1]

        resource_counts[resource_name] += 1


print("AVAILABLE DATASET RESOURCES")
print("-" * 60)

for resource, count in sorted(resource_counts.items()):
    print(f"{resource:<45} {count:>4} files")

print("-" * 60)
print("Total resources:", len(resource_counts))
print("Total download entries:", len(urls))

# ============================================================
# 1.5 IDENTIFY STAGE 1 RESOURCES
# ============================================================

stage1_keywords = [
    "cam_stereo_left_lut",
    "cam_left_labels_TMP",
    "labeltool_labels_refined",
    "calib_cam_stereo_left.json"
]

print("STAGE 1 REQUIRED RESOURCES")
print("=" * 65)

for keyword in stage1_keywords:

    matches = [
        item for item in urls
        if keyword.lower() in item["key"].lower()
    ]

    print(f"\nResource: {keyword}")
    print(f"Number of matching entries: {len(matches)}")

    for item in matches[:5]:
        print("  -", item["key"])

    if len(matches) > 5:
        print("  ... additional files omitted")



# ============================================================
# 1.6 INSPECT RGB ARCHIVE FILES
# ============================================================

TARGET_RESOURCE = "cam_stereo_left_lut"

rgb_entries = [
    item for item in urls
    if TARGET_RESOURCE in item["key"]
]

print("RGB ARCHIVE STRUCTURE")
print("=" * 65)

for index, item in enumerate(rgb_entries, start=1):

    print(f"{index:02d}. {item['key']}")

print("=" * 65)
print("Total RGB entries:", len(rgb_entries))



# ============================================================
# 1.7 INSPECT ANNOTATION RESOURCES
# ============================================================

annotation_keywords = [
    "cam_left_labels_TMP",
    "labeltool_labels_refined",
    "calib_cam_stereo_left.json"
]

for keyword in annotation_keywords:

    matches = [
        item for item in urls
        if keyword.lower() in item["key"].lower()
    ]

    print(f"\nRESOURCE: {keyword}")
    print("-" * 65)

    for item in matches:
        print(item["key"])

    print("Total entries:", len(matches))


# ============================================================
# 1.8 SELECT STAGE 1 DOWNLOAD RESOURCES
# ============================================================

# RGB camera archive: 23 split volumes + main ZIP
rgb_prefix = "SeeingThroughFog/cam_stereo_left_lut/"

rgb_entries = [
    item for item in urls
    if item["key"].startswith(rgb_prefix)
]

# Ground-truth annotations
gt_key = "SeeingThroughFog/gt_labels/cam_left_labels_TMP.zip"

gt_entries = [
    item for item in urls
    if item["key"] == gt_key
]

# Weather and lighting metadata
metadata_key = (
    "SeeingThroughFog/labeltool_labels/"
    "labeltool_labels_refined.zip"
)

metadata_entries = [
    item for item in urls
    if item["key"] == metadata_key
]

# Camera calibration
calibration_key = "SeeingThroughFog/calib_cam_stereo_left.json"

calibration_entries = [
    item for item in urls
    if item["key"] == calibration_key
]

# Combine all selected resources
selected_entries = (
    rgb_entries
    + gt_entries
    + metadata_entries
    + calibration_entries
)

print("STAGE 1 DOWNLOAD SELECTION")
print("=" * 60)

print(f"RGB archive parts: {len(rgb_entries)}")
print(f"Ground-truth files: {len(gt_entries)}")
print(f"Metadata files: {len(metadata_entries)}")
print(f"Calibration files: {len(calibration_entries)}")

print("-" * 60)
print(f"Total selected files: {len(selected_entries)}")

# Verify that all expected resources were found
assert len(rgb_entries) == 24, "Unexpected number of RGB archive parts"
assert len(gt_entries) == 1, "Ground-truth archive not found"
assert len(metadata_entries) == 1, "Metadata archive not found"
assert len(calibration_entries) == 1, "Calibration file not found"

assert len(selected_entries) == 27

print("\nSelection verified successfully!")

# ============================================================
# 1.9 DOWNLOAD FUNCTION
# ============================================================

import requests
import time

CHUNK_SIZE = 1024 * 1024  # 1 MB
MAX_RETRIES = 3

def download_file(url, destination):

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    # Final file already exists: do not download it again
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already exists: {destination.name}")
        return True

    # Temporary file used for interrupted downloads
    temporary_file = destination.with_suffix(
        destination.suffix + ".part"
    )


    for attempt in range(1, MAX_RETRIES + 1):

        try:
            # Resume from the existing partial file, if possible
            existing_size = (
                temporary_file.stat().st_size
                if temporary_file.exists()
                else 0
            )

            headers = {}

            if existing_size > 0:
                headers["Range"] = f"bytes={existing_size}-"

            print(
                f"\nDownloading: {destination.name} "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as response:

                if response.status_code == 206:
                    # Server accepted the resume request
                    content_range = response.headers.get(
                        "Content-Range", ""
                    )

                    if not content_range.startswith(
                        f"bytes {existing_size}-"
                    ):
                        raise RuntimeError(
                            "Unexpected Content-Range response."
                        )

                    mode = "ab"

                elif response.status_code == 200:
                    # Server does not support resume or starts from zero
                    mode = "wb"
                    existing_size = 0

                else:
                    response.raise_for_status()

                response.raise_for_status()

                with open(temporary_file, mode) as file:

                    for chunk in response.iter_content(
                        chunk_size=CHUNK_SIZE
                    ):
                        if chunk:
                            file.write(chunk)

            # Move completed download to its final filename
            if temporary_file.exists() and temporary_file.stat().st_size > 0:
                temporary_file.replace(destination)

                size_mb = destination.stat().st_size / (1024 ** 2)

                print(
                    f"Completed: {destination.name} "
                    f"({size_mb:.2f} MB)"
                )

                return True

            raise RuntimeError("Downloaded file is empty.")

        except Exception as error:

            print(
                f"Download failed: {type(error).__name__}: {error}"
            )

            if attempt < MAX_RETRIES:
                print("Retrying in 3 seconds...")
                time.sleep(3)

    print(f"Could not download: {destination.name}")
    return False



PROJECT CONFIGURATION
-------------------------------------------------------
Project directory: /home/maritza/projects/autonomous_vehicle
Dataset directory: /home/maritza/projects/autonomous_vehicle/dataset/seeing_through_fog
Download directory: /home/maritza/projects/autonomous_vehicle/dataset/seeing_through_fog/downloads
Extracted directory: /home/maritza/projects/autonomous_vehicle/dataset/seeing_through_fog/extracted
JSON file: /home/maritza/projects/autonomous_vehicle/download_links.json
Output directory: /home/maritza/projects/autonomous_vehicle/results/stage1_camera
DIRECTORY STATUS
-------------------------------------------------------
Downloads folder: True
Extracted folder: True
Results folder: True
DATASET METADATA
-------------------------------------------------------
Total download links: 472
JSON loaded successfully!
AVAILABLE DATASET RESOURCES
------------------------------------------------------------
calib_cam_stereo_left.json                       1 files
calib_ca

In [ ]:
# ============================================================
# 1.10 DOWNLOAD STAGE 1 DATASET
# ============================================================

download_results = []

for item in selected_entries:

    key = item["key"]
    url = item["url"]

    # Preserve original filenames
    filename = Path(key).name

    # Organize downloads into separate folders
    if key.startswith(rgb_prefix):
        destination = DOWNLOAD_DIR / "cam_stereo_left_lut" / filename

    elif key == gt_key:
        destination = DOWNLOAD_DIR / "gt_labels" / filename

    elif key == metadata_key:
        destination = DOWNLOAD_DIR / "labeltool_labels" / filename

    elif key == calibration_key:
        destination = DOWNLOAD_DIR / "calibration" / filename

    else:
        print(f"Skipping unexpected resource: {key}")
        continue

    success = download_file(url, destination)

    download_results.append({
        "resource": key,
        "destination": str(destination),
        "success": success
    })

print("\nDOWNLOAD SUMMARY")
print("=" * 60)

successful = sum(item["success"] for item in download_results)
failed = len(download_results) - successful

print(f"Successful downloads: {successful}")
print(f"Failed downloads: {failed}")
print(f"Total processed: {len(download_results)}")

# 1.11 Verify downloaded files

for folder in sorted(DOWNLOAD_DIR.iterdir()):
    if folder.is_dir():
        files = [
            f for f in folder.iterdir()
            if f.is_file() and not f.name.endswith(".part")
        ]

        total_size = sum(f.stat().st_size for f in files)

        print(f"\n{folder.name}")
        print(f"Files: {len(files)}")
        print(f"Total size: {total_size / (1024**3):.3f} GB")

partial_files = list(DOWNLOAD_DIR.rglob("*.part"))

print("\nPARTIAL DOWNLOADS")
if partial_files:
    for file in partial_files:
        print(file)
else:
    print("No partial downloads found.")


NameError: name 'selected_entries' is not defined

In [ ]:
# 1.17 Extract RGB Dataset

import subprocess

# Define archive and extraction paths
rgb_archive = DOWNLOAD_DIR / "cam_stereo_left_lut" / "cam_stereo_left_lut.zip"

rgb_extract_dir = EXTRACTED_DIR / "cam_stereo_left_lut"

# Create extraction directory
rgb_extract_dir.mkdir(parents=True, exist_ok=True)

print("RGB DATASET EXTRACTION")
print("=" * 60)

print(f"Archive: {rgb_archive}")
print(f"Destination: {rgb_extract_dir}")

# Extract multivolume ZIP archive using 7-Zip
result = subprocess.run(
    [
        "7z",
        "x",
        str(rgb_archive),
        f"-o{rgb_extract_dir}",
        "-y"
    ],
    capture_output=True,
    text=True
)

print("\nExtraction process finished.")
print(f"Return code: {result.returncode}")

if result.returncode == 0:
    print("Extraction completed successfully.")
else:
    print("Extraction encountered an error.")
    print(result.stderr[-3000:])

# 1.18 Verify Extracted RGB Dataset

from collections import Counter

# Define the extracted RGB dataset directory
rgb_extract_dir = EXTRACTED_DIR / "cam_stereo_left_lut"

# Find all extracted files
extracted_files = [
    path for path in rgb_extract_dir.rglob("*")
    if path.is_file()
]

# Count files by extension
extension_counts = Counter(
    path.suffix.lower() for path in extracted_files
)

# Calculate total extracted size
total_size_bytes = sum(
    path.stat().st_size for path in extracted_files
)

total_size_gb = total_size_bytes / (1024**3)

print("EXTRACTED RGB DATASET VERIFICATION")
print("=" * 60)

print(f"Extraction directory: {rgb_extract_dir}")
print(f"Total extracted files: {len(extracted_files)}")
print(f"Total extracted size: {total_size_gb:.2f} GB")

print("\nFILE TYPES")
print("-" * 60)

for extension, count in extension_counts.most_common():
    print(f"{extension or '[no extension]'}: {count} files")

# Check whether the extraction directory contains files
if extracted_files:
    print("\nVerification result: Files found.")
else:
    print("\nVerification result: No files found.")

In [ ]:
# 2.19 — Inventory All ZIP Archives

from pathlib import Path

DOWNLOAD_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "seeing_through_fog"
    / "downloads"
)

EXTRACTED_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "seeing_through_fog"
    / "extracted"
)

zip_archives = sorted(DOWNLOAD_DIR.rglob("*.zip"))

print("DOWNLOADED ZIP ARCHIVES")
print("=" * 75)
print(f"Total ZIP archives found: {len(zip_archives)}")

for archive_path in zip_archives:

    relative_path = archive_path.relative_to(DOWNLOAD_DIR)

    # Each archive is extracted into a folder named after its parent folder
    extraction_dir = EXTRACTED_ROOT / relative_path.parent

    already_extracted = (
        extraction_dir.exists()
        and any(path.is_file() for path in extraction_dir.rglob("*"))
    )

    size_gb = archive_path.stat().st_size / (1024 ** 3)

    print(f"\nArchive: {relative_path}")
    print(f"Size: {size_gb:.3f} GB")
    print(f"Extraction directory: {extraction_dir}")
    print(f"Already extracted: {already_extracted}")

# 2.20 — Extract All Remaining ZIP Archives

import zipfile

print("EXTRACTING REMAINING DATASET ARCHIVES")
print("=" * 75)

for archive_path in zip_archives:

    relative_path = archive_path.relative_to(DOWNLOAD_DIR)
    extraction_dir = EXTRACTED_ROOT / relative_path.parent

    already_extracted = (
        extraction_dir.exists()
        and any(path.is_file() for path in extraction_dir.rglob("*"))
    )

    print(f"\nArchive: {relative_path}")

    if already_extracted:
        print("Status: Skipped — files already extracted.")
        continue

    if not zipfile.is_zipfile(archive_path):
        print("Status: Skipped — not a standard ZIP archive.")
        continue

    extraction_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(archive_path, "r") as archive:

            archive.extractall(extraction_dir)

            extracted_count = sum(
                1 for name in archive.namelist()
                if not name.endswith("/")
            )

        print("Status: Extraction completed.")
        print(f"Files extracted: {extracted_count}")
        print(f"Destination: {extraction_dir}")

    except Exception as error:
        print(f"Status: Extraction failed — {error}")

print("\nEXTRACTION PROCESS FINISHED")


In [3]:
# ============================================================
# 1.12. INSPECT LIDAR AND SENSOR CALIBRATION RESOURCES
# ============================================================

# Identify LiDAR-related resources in the complete dataset catalog.
lidar_keywords = [
    "lidar",
    "hdl64",
    "vlp32"
]

lidar_entries = [
    item for item in urls
    if any(
        keyword in item["key"].lower()
        for keyword in lidar_keywords
    )
]

# Identify sensor calibration resources.
calibration_entries_all = [
    item for item in urls
    if "calib" in item["key"].lower()
    or "tf_tree" in item["key"].lower()
]

print("LIDAR RESOURCES")
print("=" * 75)

for item in lidar_entries:
    print(item["key"])

print("\nTotal LiDAR entries:", len(lidar_entries))

print("\nSENSOR CALIBRATION RESOURCES")
print("=" * 75)

for item in calibration_entries_all:
    print(item["key"])

print("\nTotal calibration entries:", len(calibration_entries_all))

LIDAR RESOURCES
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z01
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z02
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z03
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z04
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z05
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z06
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.zip
SeeingThroughFog/lidar_hdl64_last_gated/lidar_hdl64_last_gated.zip
SeeingThroughFog/lidar_hdl64_last_stereo_left/lidar_hdl64_last_stereo_left.zip
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z01
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z02
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z03
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z04
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z05
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.z06
SeeingThroughFog/lidar_hdl64_strongest/lidar_hdl64_strongest.zi

In [9]:
# ============================================================
# 1.13. INSPECT SELECTED LIDAR RESOURCES
# ============================================================

TARGET_RESOURCES = [
    "lidar_hdl64_last",
    "lidar_hdl64_last_stereo_left",
    "calib_tf_tree_full.json"
]

for resource in TARGET_RESOURCES:

    print("\n" + "=" * 75)
    print(f"RESOURCE: {resource}")
    print("=" * 75)

    matches = [
        item for item in urls
        if resource.lower() in item["key"].lower()
    ]

    for item in matches:
        print(item["key"])

        # Inspect available metadata fields
        print("Metadata fields:", list(item.keys()))

    print("Total files:", len(matches))


RESOURCE: lidar_hdl64_last
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z01
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z02
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z03
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z04
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z05
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z06
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.zip
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last_gated/lidar_hdl64_last_gated.zip
Metadata fields: ['key', 'url']
SeeingThroughFog/lidar_hdl64_last_stereo_left/lidar_hdl64_last_stereo_left.zip
Metadata fields: ['key', 'url']
Total files: 9

RESOURCE: lidar_hdl64_last_stereo_left
SeeingThroughFog/lidar_hdl64_last_stereo_left/lidar_hdl64_last_stereo_left.zip
Metadata fields: ['

In [10]:
# ============================================================
# 1.14. SELECT LIDAR AND CALIBRATION DOWNLOADS
# ============================================================

# Original HDL64 LiDAR archive (split ZIP)
lidar_prefix = "SeeingThroughFog/lidar_hdl64_last/"

lidar_entries = [
    item for item in urls
    if item["key"].startswith(lidar_prefix)
]

# LiDAR representation projected onto the left stereo camera
lidar_projection_key = (
    "SeeingThroughFog/lidar_hdl64_last_stereo_left/"
    "lidar_hdl64_last_stereo_left.zip"
)

lidar_projection_entries = [
    item for item in urls
    if item["key"] == lidar_projection_key
]

# Full sensor transformation tree
tf_calibration_key = "SeeingThroughFog/calib_tf_tree_full.json"

tf_calibration_entries = [
    item for item in urls
    if item["key"] == tf_calibration_key
]

# Combine selected resources
lidar_selected_entries = (
    lidar_entries
    + lidar_projection_entries
    + tf_calibration_entries
)

print("LIDAR DOWNLOAD SELECTION")
print("=" * 65)

print(f"Original LiDAR archive parts: {len(lidar_entries)}")
print(f"Projected LiDAR archive: {len(lidar_projection_entries)}")
print(f"Sensor calibration files: {len(tf_calibration_entries)}")
print(f"Total selected files: {len(lidar_selected_entries)}")

for item in lidar_selected_entries:
    print("-", item["key"])

# Verify expected resources
assert len(lidar_entries) == 7, "Unexpected LiDAR archive structure"
assert len(lidar_projection_entries) == 1, "Projected LiDAR archive not found"
assert len(tf_calibration_entries) == 1, "TF calibration file not found"

print("\nSelection verified successfully!")

LIDAR DOWNLOAD SELECTION
Original LiDAR archive parts: 7
Projected LiDAR archive: 1
Sensor calibration files: 1
Total selected files: 9
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z01
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z02
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z03
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z04
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z05
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z06
- SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.zip
- SeeingThroughFog/lidar_hdl64_last_stereo_left/lidar_hdl64_last_stereo_left.zip
- SeeingThroughFog/calib_tf_tree_full.json

Selection verified successfully!


In [11]:
# ============================================================
# 1.15. DOWNLOAD LIDAR AND SENSOR CALIBRATION
# ============================================================

lidar_download_results = []

for item in lidar_selected_entries:

    key = item["key"]
    url = item["url"]

    filename = Path(key).name

    # Organize each resource in its own directory
    if key.startswith(lidar_prefix):
        destination = (
            DOWNLOAD_DIR / "lidar_hdl64_last" / filename
        )

    elif key == lidar_projection_key:
        destination = (
            DOWNLOAD_DIR
            / "lidar_hdl64_last_stereo_left"
            / filename
        )

    elif key == tf_calibration_key:
        destination = (
            DOWNLOAD_DIR / "calibration" / filename
        )

    else:
        print(f"Unexpected resource: {key}")
        continue

    success = download_file(url, destination)

    lidar_download_results.append({
        "resource": key,
        "destination": str(destination),
        "success": success
    })

# Download summary
successful = sum(
    item["success"] for item in lidar_download_results
)

failed = len(lidar_download_results) - successful

print("\nLIDAR DOWNLOAD SUMMARY")
print("=" * 65)
print(f"Successful: {successful}")
print(f"Failed: {failed}")
print(f"Total: {len(lidar_download_results)}")

for item in lidar_download_results:
    print(
        "OK" if item["success"] else "FAILED",
        item["resource"]
    )


Downloading: lidar_hdl64_last.z01 (attempt 1/3)
Completed: lidar_hdl64_last.z01 (1500.00 MB)

Downloading: lidar_hdl64_last.z02 (attempt 1/3)
Completed: lidar_hdl64_last.z02 (1500.00 MB)

Downloading: lidar_hdl64_last.z03 (attempt 1/3)
Completed: lidar_hdl64_last.z03 (1500.00 MB)

Downloading: lidar_hdl64_last.z04 (attempt 1/3)
Completed: lidar_hdl64_last.z04 (1500.00 MB)

Downloading: lidar_hdl64_last.z05 (attempt 1/3)
Completed: lidar_hdl64_last.z05 (1500.00 MB)

Downloading: lidar_hdl64_last.z06 (attempt 1/3)
Completed: lidar_hdl64_last.z06 (1500.00 MB)

Downloading: lidar_hdl64_last.zip (attempt 1/3)
Completed: lidar_hdl64_last.zip (4.11 MB)

Downloading: lidar_hdl64_last_stereo_left.zip (attempt 1/3)
Completed: lidar_hdl64_last_stereo_left.zip (372.74 MB)

Downloading: calib_tf_tree_full.json (attempt 1/3)
Completed: calib_tf_tree_full.json (0.02 MB)

LIDAR DOWNLOAD SUMMARY
Successful: 9
Failed: 0
Total: 9
OK SeeingThroughFog/lidar_hdl64_last/lidar_hdl64_last.z01
OK SeeingThrough

In [17]:
# ============================================================
# 2.1. LIDAR DATASET CONFIGURATION
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()

EXTRACTED_DIR = (
    PROJECT_DIR
    / "dataset"
    / "seeing_through_fog"
    / "extracted"
)

# Actual directories containing the extracted files
LIDAR_DIR = (
    EXTRACTED_DIR
    / "lidar_hdl64_last"
    / "lidar_hdl64_last"
)

LIDAR_PROJECTED_DIR = (
    EXTRACTED_DIR
    / "lidar_hdl64_last_stereo_left"
    / "lidar_hdl64_last_stereo_left"
)

CAMERA_DIR = (
    EXTRACTED_DIR
    / "cam_stereo_left_lut"
    / "cam_stereo_left_lut"
)

CALIBRATION_PATH = (
    PROJECT_DIR
    / "dataset"
    / "seeing_through_fog"
    / "downloads"
    / "calibration"
    / "calib_tf_tree_full.json"
)

print("LIDAR DATASET CONFIGURATION")
print("-" * 60)

print("Original LiDAR:", LIDAR_DIR.exists())
print("Projected LiDAR:", LIDAR_PROJECTED_DIR.exists())
print("RGB camera:", CAMERA_DIR.exists())
print("Calibration:", CALIBRATION_PATH.exists())

# ============================================================
# 2.2. INSPECT LIDAR FILE STRUCTURE
# ============================================================

lidar_files = sorted([
    path for path in LIDAR_DIR.rglob("*")
    if path.is_file()
])

projected_files = sorted([
    path for path in LIDAR_PROJECTED_DIR.rglob("*")
    if path.is_file()
])

print("ORIGINAL LIDAR FILES")
print("-" * 60)

for path in lidar_files[:10]:
    print(path.name, "|", path.stat().st_size, "bytes")

print("\nPROJECTED LIDAR FILES")
print("-" * 60)

for path in projected_files[:10]:
    print(path.name, "|", path.stat().st_size, "bytes")

print("\nTotal original files:", len(lidar_files))
print("Total projected files:", len(projected_files))

LIDAR DATASET CONFIGURATION
------------------------------------------------------------
Original LiDAR: True
Projected LiDAR: True
RGB camera: True
Calibration: True
ORIGINAL LIDAR FILES
------------------------------------------------------------
2018-02-03_20-48-35_00400.bin | 2210280 bytes
2018-02-03_20-48-35_00500.bin | 2204120 bytes
2018-02-03_20-49-38_00000.bin | 2208900 bytes
2018-02-03_20-49-38_00100.bin | 2209020 bytes
2018-02-03_20-49-38_00200.bin | 2209980 bytes
2018-02-03_20-50-46_00000.bin | 2208980 bytes
2018-02-03_20-50-46_00100.bin | 2202160 bytes
2018-02-03_20-50-46_00200.bin | 2202420 bytes
2018-02-03_20-52-49_00000.bin | 2214860 bytes
2018-02-03_20-53-15_00000.bin | 2208660 bytes

PROJECTED LIDAR FILES
------------------------------------------------------------
2018-02-03_20-48-35_00400.npz | 56087 bytes
2018-02-03_20-48-35_00500.npz | 57097 bytes
2018-02-03_20-49-38_00000.npz | 56742 bytes
2018-02-03_20-49-38_00100.npz | 59305 bytes
2018-02-03_20-49-38_00200.npz |

In [18]:
# ============================================================
# 2.3. INSPECT LIDAR FILE FORMATS
# ============================================================

sample_name = "2018-02-03_20-48-35_00400"

original_file = LIDAR_DIR / f"{sample_name}.bin"
projected_file = LIDAR_PROJECTED_DIR / f"{sample_name}.npz"

print("ORIGINAL LIDAR FILE")
print("-" * 60)

file_size = original_file.stat().st_size

print("File size:", file_size, "bytes")
print("Divisible by 4:", file_size % 4 == 0)
print("Divisible by 16:", file_size % 16 == 0)

with open(original_file, "rb") as file:
    raw_sample = file.read(64)

print("First 64 bytes:")
print(raw_sample.hex(" "))

print("\nPROJECTED LIDAR FILE")
print("-" * 60)

with np.load(projected_file, allow_pickle=False) as data:

    print("Available arrays:", data.files)

    for key in data.files:

        array = data[key]

        print(f"\nArray: {key}")
        print("Shape:", array.shape)
        print("Data type:", array.dtype)
        print("Dimensions:", array.ndim)

        if array.size > 0:
            print("First values:")
            print(array.reshape(-1)[:10])

ORIGINAL LIDAR FILE
------------------------------------------------------------
File size: 2210280 bytes
Divisible by 4: True
Divisible by 16: False
First 64 bytes:
9f 38 dc 40 89 9f 0d 41 99 94 9a bf 00 00 7f 43 00 00 10 42 35 ee ce 40 b6 3b 0e 41 8f 2e 90 bf 00 00 73 43 00 00 14 42 9f c9 a3 40 03 9a 0d 41 c8 81 78 3e 66 66 77 43 00 00 68 42 f0 7f 97 40

PROJECTED LIDAR FILE
------------------------------------------------------------
Available arrays: ['arr_0']

Array: arr_0
Shape: (1024, 1920)
Data type: float32
Dimensions: 2
First values:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [19]:
# ============================================================
# 2.4. LIDAR DATA STRUCTURE ANALYSIS
# ============================================================

# Load original LiDAR binary data
raw_lidar = np.fromfile(original_file, dtype=np.float32)

print("ORIGINAL LIDAR")
print("-" * 60)
print("Total float32 values:", len(raw_lidar))
print("Values per point candidate:", 5)

# Exploratory interpretation: five float32 values per record
if len(raw_lidar) % 5 == 0:

    lidar_points = raw_lidar.reshape(-1, 5)

    print("Point cloud shape:", lidar_points.shape)

    for column in range(lidar_points.shape[1]):

        values = lidar_points[:, column]
        finite_values = values[np.isfinite(values)]

        print(f"\nColumn {column}")
        print("Finite values:", len(finite_values))
        print("Minimum:", np.min(finite_values))
        print("Maximum:", np.max(finite_values))
        print("Mean:", np.mean(finite_values))
        print("Standard deviation:", np.std(finite_values))
        print("First 10 values:", finite_values[:10])


# Inspect projected LiDAR matrix
with np.load(projected_file, allow_pickle=False) as data:

    projected_lidar = data["arr_0"]

print("\nPROJECTED LIDAR")
print("-" * 60)

valid_mask = np.isfinite(projected_lidar) & (projected_lidar != 0)

print("Shape:", projected_lidar.shape)
print("Data type:", projected_lidar.dtype)
print("Total pixels:", projected_lidar.size)
print("Non-zero finite pixels:", np.count_nonzero(valid_mask))
print("Percentage non-zero:", 100 * valid_mask.mean())

if np.any(valid_mask):

    valid_values = projected_lidar[valid_mask]

    print("Minimum non-zero value:", valid_values.min())
    print("Maximum non-zero value:", valid_values.max())
    print("Mean non-zero value:", valid_values.mean())
    print("Median non-zero value:", np.median(valid_values))
    print("Standard deviation:", valid_values.std())
    print("First 20 non-zero values:", valid_values[:20])

ORIGINAL LIDAR
------------------------------------------------------------
Total float32 values: 552570
Values per point candidate: 5
Point cloud shape: (110514, 5)

Column 0
Finite values: 110514
Minimum: -109.048294
Maximum: 109.91262
Mean: 1.0764425
Standard deviation: 13.791228
First 10 values: [6.8819118 6.466578  5.118362  4.7343674 5.887015  5.41981   6.0478168
 5.6473756 4.9865236 4.558451 ]

Column 1
Finite values: 110514
Minimum: -80.57042
Maximum: 42.35243
Mean: -0.47048
Standard deviation: 7.287289
First 10 values: [8.851449  8.889578  8.8501005 8.868634  8.8750725 8.81372   8.865077
 8.896428  8.901266  8.814941 ]

Column 2
Finite values: 110514
Minimum: -36.35103
Maximum: 3.764115
Mean: -0.9101641
Standard deviation: 1.5345196
First 10 values: [-1.2076598  -1.1264209   0.24268258  0.2889549  -1.0242202  -0.9219293
 -1.3874649  -1.3179584  -0.84010375 -0.7570745 ]

Column 3
Finite values: 110514
Minimum: 24.0
Maximum: 255.0
Mean: 150.13991
Standard deviation: 75.75034
Fir

In [20]:
# ============================================================
# 2.5. DETAILED LIDAR FEATURE INSPECTION
# ============================================================

lidar_points = np.fromfile(
    original_file,
    dtype=np.float32
).reshape(-1, 5)

column_names = [
    "Column 0",
    "Column 1",
    "Column 2",
    "Column 3",
    "Column 4"
]

for i, name in enumerate(column_names):

    values = lidar_points[:, i]

    print(f"\n{name}")
    print("-" * 40)
    print("Minimum:", float(values.min()))
    print("Maximum:", float(values.max()))
    print("Mean:", float(values.mean()))
    print("Std:", float(values.std()))
    print("First 15:", values[:15])


Column 0
----------------------------------------
Minimum: -109.04829406738281
Maximum: 109.9126205444336
Mean: 1.0764424800872803
Std: 13.791228294372559
First 15: [6.8819118 6.466578  5.118362  4.7343674 5.887015  5.41981   6.0478168
 5.6473756 4.9865236 4.558451  5.1015577 4.7311616 6.8993735 6.389171
 7.056923 ]

Column 1
----------------------------------------
Minimum: -80.57041931152344
Maximum: 42.3524284362793
Mean: -0.4704799950122833
Std: 7.287289142608643
First 15: [8.851449  8.889578  8.8501005 8.868634  8.8750725 8.81372   8.865077
 8.896428  8.901266  8.814941  8.818843  8.848984  8.870475  8.842183
 8.850272 ]

Column 2
----------------------------------------
Minimum: -36.35102844238281
Maximum: 3.76411509513855
Mean: -0.9101641178131104
Std: 1.5345195531845093
First 15: [-1.2076598  -1.1264209   0.24268258  0.2889549  -1.0242202  -0.9219293
 -1.3874649  -1.3179584  -0.84010375 -0.7570745  -1.2072903  -1.1188874
 -0.4099238  -0.33208174 -0.8180148 ]

Column 3
--------

In [21]:
# ============================================================
# 2.6. PROJECTED LIDAR VALUE DISTRIBUTION
# ============================================================

with np.load(projected_file, allow_pickle=False) as data:
    projected_lidar = data["arr_0"]

valid_mask = (
    np.isfinite(projected_lidar)
    & (projected_lidar > 0)
)

valid_values = projected_lidar[valid_mask]

print("PROJECTED LIDAR DISTRIBUTION")
print("-" * 50)

print("Valid pixels:", len(valid_values))
print("Zero pixels:", np.count_nonzero(projected_lidar == 0))
print("NaN pixels:", np.count_nonzero(np.isnan(projected_lidar)))

print("\nPercentiles:")

for percentile in [0, 10, 25, 50, 75, 90, 95, 99, 100]:

    print(
        f"{percentile}%:",
        float(np.percentile(valid_values, percentile))
    )

PROJECTED LIDAR DISTRIBUTION
--------------------------------------------------
Valid pixels: 8618
Zero pixels: 1957462
NaN pixels: 0

Percentiles:
0%: 1.5773807764053345
10%: 6.7788801193237305
25%: 9.12015151977539
50%: 15.543563842773438
75%: 25.055343627929688
90%: 50.09665298461914
95%: 80.2827377319336
99%: 94.75346374511719
100%: 109.17823028564453


In [22]:
# ============================================================
# 2.7. INSPECT REMAINING LIDAR FEATURES
# ============================================================

for i in [2, 3, 4]:

    values = lidar_points[:, i]

    print(f"\nCOLUMN {i}")
    print("-" * 45)
    print("Minimum:", float(values.min()))
    print("Maximum:", float(values.max()))
    print("Mean:", float(values.mean()))
    print("Std:", float(values.std()))
    print("First 20 values:", values[:20])

print("\nFIRST 10 COMPLETE LIDAR RECORDS")
print("-" * 45)
print(lidar_points[:10])


COLUMN 2
---------------------------------------------
Minimum: -36.35102844238281
Maximum: 3.76411509513855
Mean: -0.9101641178131104
Std: 1.5345195531845093
First 20 values: [-1.2076598  -1.1264209   0.24268258  0.2889549  -1.0242202  -0.9219293
 -1.3874649  -1.3179584  -0.84010375 -0.7570745  -1.2072903  -1.1188874
 -0.4099238  -0.33208174 -0.8180148  -0.73322666 -0.24909186 -0.18705812
 -0.6388385  -0.56893003]

COLUMN 3
---------------------------------------------
Minimum: 24.0
Maximum: 255.0
Mean: 150.13990783691406
Std: 75.7503433227539
First 20 values: [255.  243.  247.4 255.  255.  255.  159.  255.  243.8 255.  255.  255.
 255.  251.9 255.  255.  255.  255.  255.  255. ]

COLUMN 4
---------------------------------------------
Minimum: 0.0
Maximum: 63.0
Mean: 31.416744232177734
Std: 18.453651428222656
First 20 values: [36. 37. 58. 59. 38. 39. 32. 33. 40. 41. 34. 35. 48. 49. 42. 43. 50. 51.
 44. 45.]

FIRST 10 COMPLETE LIDAR RECORDS
--------------------------------------------